In [4]:
import re
import polars as pl
from bs4 import BeautifulSoup
from pathlib import Path
from datetime import datetime
from tqdm.auto import tqdm

BRONZE_DIR = Path("../data/bronze")
SILVER_DIR = Path("../data/silver")
SILVER_DIR.mkdir(parents=True, exist_ok=True)

def parse_items(soup):
    """Heuristic extraction of item names and gold costs."""
    items = []
    
    tags = soup.find_all(string=re.compile(r"(\d+)\s*(Gold|Cost)", re.IGNORECASE))
    
    for tag in tags:
        text = tag.parent.get_text().strip()
        
        match = re.search(r"^(.*?)\s*[:\-\(]?\s*(\d+)\s*(Gold|Cost)", text, re.IGNORECASE)
        
        if match:
            name = match.group(1).strip()
            name = re.sub(r"^[•\-\*]\s*", "", name)
            cost = int(match.group(2))
            
            if name and 2 < len(name) < 50:
                items.append({
                    "item_name": name,
                    "gold_cost": cost
                })
                
    return items



print(f"Starting ingestion: {BRONZE_DIR} -> {SILVER_DIR}")

folders = sorted([f for f in BRONZE_DIR.iterdir() if f.is_dir()])
stats = {"success": 0, "empty": 0, "total_items": 0, "no_html": 0}

for folder in tqdm(folders, desc="Processing patches"):
    patch_id = folder.name
    
    html_files = list(folder.rglob("*.html")) + list(folder.rglob("*.HTML"))
    
    if not html_files:
        print(f"Skip '{patch_id}': No HTML found.")
        stats["no_html"] += 1
        continue
        
    source_file = html_files[0]
    
    try:
        with open(source_file, "r", encoding="utf-8") as f:
            soup = BeautifulSoup(f, "lxml")
            
        extracted = parse_items(soup)
        
        if extracted:
            df = pl.DataFrame(extracted).unique(subset=["item_name"])
            
            df = df.with_columns([
                pl.lit(patch_id).alias("patch_version"),
                pl.lit(source_file.name).alias("source_file"),
                pl.lit(datetime.now()).alias("ingested_at")
            ])
            
            out_name = f"patch_{patch_id.replace('.', '-')}.parquet"
            df.write_parquet(SILVER_DIR / out_name)
            
            stats["success"] += 1
            stats["total_items"] += len(df)
        else:
            print(f"Info: '{patch_id}' - HTML found but no items extracted.")
            stats["empty"] += 1
            
    except Exception as e:
        print(f"Error processing {patch_id}: {e}")

print("\n" + "-"*30)
print("Ingestion Report")
print(f"Patches processed:  {stats['success']}")
print(f"Patches skipped:    {stats['no_html']}")
print(f"Empty results:      {stats['empty']}")
print(f"Total items saved:  {stats['total_items']}")
print("-"*30)

Starting ingestion: ../data/bronze -> ../data/silver


Processing patches:   0%|          | 0/4 [00:00<?, ?it/s]

Processing patches:  25%|██▌       | 1/4 [00:00<00:00,  8.52it/s]

Info: 'alpha' - HTML found but no items extracted.


Processing patches:  50%|█████     | 2/4 [00:00<00:00,  6.13it/s]

Info: 'beta' - HTML found but no items extracted.


Processing patches:  75%|███████▌  | 3/4 [00:00<00:00,  3.97it/s]

Info: 'season_one' - HTML found but no items extracted.


Processing patches: 100%|██████████| 4/4 [00:00<00:00,  4.41it/s]

Info: 'season_two' - HTML found but no items extracted.

------------------------------
Ingestion Report
Patches processed:  0
Patches skipped:    0
Empty results:      4
Total items saved:  0
------------------------------
